<a href="https://colab.research.google.com/github/Numanur/heart-failure-monitoring-llm-rag/blob/main/Thesis_03_corpus_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
!pip install -q pymupdf pandas numpy tqdm

In [24]:
import os
import re
import json
from pathlib import Path
from datetime import datetime

import fitz  # PyMuPDF
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

PROJECT_DIR = "/content/drive/MyDrive/llm"

RAW_CORPUS_DIR = f"{PROJECT_DIR}/rag_corpus_raw"
PROCESSED_CORPUS_DIR = f"{PROJECT_DIR}/rag_corpus_processed"

os.makedirs(RAW_CORPUS_DIR, exist_ok=True)
os.makedirs(PROCESSED_CORPUS_DIR, exist_ok=True)

DOCUMENT_INVENTORY_PATH = f"{PROCESSED_CORPUS_DIR}/document_inventory.csv"
EXTRACTED_PAGES_PATH = f"{PROCESSED_CORPUS_DIR}/extracted_document_pages.csv"
CLEANED_PAGES_PATH = f"{PROCESSED_CORPUS_DIR}/cleaned_document_pages.csv"
PREPARATION_REPORT_PATH = f"{PROCESSED_CORPUS_DIR}/corpus_preparation_report.json"

print("Project directory:", PROJECT_DIR)
print("Raw corpus directory:", RAW_CORPUS_DIR)
print("Processed corpus directory:", PROCESSED_CORPUS_DIR)

Project directory: /content/drive/MyDrive/llm
Raw corpus directory: /content/drive/MyDrive/llm/rag_corpus_raw
Processed corpus directory: /content/drive/MyDrive/llm/rag_corpus_processed


In [26]:
pdf_files = sorted(Path(RAW_CORPUS_DIR).rglob("*.pdf"))

print("PDF files found:", len(pdf_files))

for i, path in enumerate(pdf_files, start=1):
    print(f"{i}. {path.name}")

if len(pdf_files) == 0:
    raise FileNotFoundError(
        f"No PDF files found in {RAW_CORPUS_DIR}. "
        "Please upload your 4 heart-failure PDFs into this folder."
    )

PDF files found: 4
1. 1 heidenreich-et-al-2022-2022-aha-acc-hfsa-guideline-for-the-management-of-heart-failure-a-report-of-the-american-college.pdf
2. 2 chronic-heart-failure-in-adults-diagnosis-and-management-pdf-66141541311685.pdf
3. 3 acute-heart-failure-diagnosis-and-management-pdf-35109817738693.pdf
4. 4 DS18660_ENG_Patient-Discharge-Packet_2022.pdf


In [ ]:
def infer_document_metadata(pdf_path, fallback_index):
    file_name = pdf_path.name
    lower_name = file_name.lower()

    if "heidenreich" in lower_name or "aha-acc-hfsa" in lower_name or "acc-hfsa" in lower_name:
        return {
            "document_id": "DOC001",
            "file_name": file_name,
            "document_name": "2022 AHA/ACC/HFSA Guideline for the Management of Heart Failure",
            "source_name": "AHA/ACC/HFSA",
            "document_type": "clinical_guideline",
            "audience": "clinician",
            "year": "2022",
            "trust_level": "high",
            "priority": "high",
            "main_use": "Clinical HF management, HF classification, GDMT, decongestion, hospitalization, transitions of care"
        }

    if "chronic-heart-failure" in lower_name or "ng106" in lower_name:
        return {
            "document_id": "DOC002",
            "file_name": file_name,
            "document_name": "NICE Chronic Heart Failure in Adults: Diagnosis and Management",
            "source_name": "NICE",
            "document_type": "chronic_hf_guideline",
            "audience": "clinician_nurse",
            "year": "2018_updated_2025",
            "trust_level": "high",
            "priority": "high",
            "main_use": "Chronic HF management, care planning, follow-up, medication monitoring"
        }

    if "acute-heart-failure" in lower_name or "cg187" in lower_name:
        return {
            "document_id": "DOC003",
            "file_name": file_name,
            "document_name": "NICE Acute Heart Failure: Diagnosis and Management",
            "source_name": "NICE",
            "document_type": "acute_hf_guideline",
            "audience": "clinician_nurse",
            "year": "2014_updated_2021",
            "trust_level": "high",
            "priority": "high",
            "main_use": "Acute HF diagnosis, stabilization, treatment after stabilization, discharge transition"
        }

    if "ds18660" in lower_name or "discharge-packet" in lower_name or "discharge" in lower_name:
        return {
            "document_id": "DOC004",
            "file_name": file_name,
            "document_name": "AHA Discharge Packet for Patients Diagnosed with Heart Failure",
            "source_name": "AHA",
            "document_type": "patient_discharge_education",
            "audience": "patient_nurse",
            "year": "2022",
            "trust_level": "high",
            "priority": "medium",
            "main_use": "Patient education, self-care, medication adherence, symptom warning signs, discharge counseling"
        }

    return {
        "document_id": f"DOC{fallback_index:03d}",
        "file_name": file_name,
        "document_name": pdf_path.stem,
        "source_name": "unknown",
        "document_type": "unknown",
        "audience": "unknown",
        "year": "unknown",
        "trust_level": "review",
        "priority": "review",
        "main_use": "manual_review_needed"
    }


inventory_rows = []
used_ids = set()

for idx, pdf_path in enumerate(pdf_files, start=1):
    meta = infer_document_metadata(pdf_path, idx)

    if meta["document_id"] in used_ids:
        meta["document_id"] = f"{meta['document_id']}_DUP{idx}"

    used_ids.add(meta["document_id"])

    meta["source_path"] = str(pdf_path)
    meta["file_size_mb"] = round(os.path.getsize(pdf_path) / (1024 * 1024), 3)
    meta["created_at"] = datetime.now().isoformat()

    inventory_rows.append(meta)

document_inventory = pd.DataFrame(inventory_rows)

document_inventory = document_inventory.sort_values("document_id").reset_index(drop=True)

document_inventory.to_csv(DOCUMENT_INVENTORY_PATH, index=False)

display(document_inventory)

print("Saved:", DOCUMENT_INVENTORY_PATH)

In [28]:
expected_doc_ids = {"DOC001", "DOC002", "DOC003", "DOC004"}
found_doc_ids = set(document_inventory["document_id"].tolist())

missing_doc_ids = expected_doc_ids - found_doc_ids

if missing_doc_ids:
    print("Warning: expected document IDs missing:", missing_doc_ids)
    print("Found document IDs:", found_doc_ids)
else:
    print("All 4 expected prototype documents are present.")

All 4 expected prototype documents are present.


In [ ]:
def extract_pdf_pages_with_pymupdf(pdf_path, document_id, file_name):
    rows = []

    try:
        doc = fitz.open(str(pdf_path))
    except Exception as e:
        raise RuntimeError(f"Could not open PDF: {pdf_path}. Error: {e}")

    for page_index in range(len(doc)):
        page_number = page_index + 1
        page = doc[page_index]

        try:
            raw_text = page.get_text("text", sort=True)
        except TypeError:
            raw_text = page.get_text("text")

        if raw_text is None:
            raw_text = ""

        rows.append({
            "document_id": document_id,
            "file_name": file_name,
            "page_number": page_number,
            "raw_text": raw_text,
            "raw_char_count": len(raw_text),
            "raw_word_count": len(raw_text.split())
        })

    doc.close()
    return rows


all_page_rows = []

for _, row in tqdm(document_inventory.iterrows(), total=len(document_inventory)):
    pdf_path = Path(row["source_path"])

    page_rows = extract_pdf_pages_with_pymupdf(
        pdf_path=pdf_path,
        document_id=row["document_id"],
        file_name=row["file_name"]
    )

    all_page_rows.extend(page_rows)

extracted_pages = pd.DataFrame(all_page_rows)

extracted_pages.to_csv(EXTRACTED_PAGES_PATH, index=False)

print("Extracted page rows:", extracted_pages.shape)
display(extracted_pages.head())

print("Saved:", EXTRACTED_PAGES_PATH)

In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    replacements = {
        "\x00": " ",
        "\ufffe": "",
        "￾": "",
        "–": "-",
        "—": "-",
        "“": '"',
        "”": '"',
        "’": "'",
        "‘": "'",
        "\u00a0": " "
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    # Fix words broken across line breaks: manage-\nment -> management
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    return text


def is_noise_line(line, document_id):
    low = line.lower().strip()

    if not low:
        return True

    if low.startswith("downloaded from "):
        return True

    if "© nice" in low:
        return True

    if "subject to notice of rights" in low:
        return True

    if "page " in low and re.fullmatch(r"page \d+ of", low):
        return True

    if re.fullmatch(r"\d+", low):
        return True

    if "chronic heart failure in adults: diagnosis and management" in low:
        return True

    if "acute heart failure: diagnosis and management" in low:
        return True

    if "heidenreich et al" in low and "heart failure guideline" in low:
        return True

    if low in ["clinical statements", "and guidelines"]:
        return True

    if "circulation. 2022;145" in low:
        return True

    if "we have many other fact sheets" in low:
        return True

    if "visit heart.org/answersbyheart" in low:
        return True

    if low in ["how can i learn more?", "my questions:"]:
        return True

    return False


def clean_page_text(text, document_id):
    text = normalize_text(text)

    cleaned_lines = []

    for line in text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()

        if is_noise_line(line, document_id):
            continue

        cleaned_lines.append(line)

    cleaned_text = "\n".join(cleaned_lines)
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)

    return cleaned_text.strip()


cleaned_pages = extracted_pages.copy()

cleaned_pages["clean_text"] = cleaned_pages.apply(
    lambda row: clean_page_text(row["raw_text"], row["document_id"]),
    axis=1
)

cleaned_pages["clean_char_count"] = cleaned_pages["clean_text"].apply(len)
cleaned_pages["clean_word_count"] = cleaned_pages["clean_text"].apply(lambda x: len(str(x).split()))

cleaned_pages.to_csv(CLEANED_PAGES_PATH, index=False)

print("Cleaned page rows:", cleaned_pages.shape)
display(cleaned_pages.head())

print("Saved:", CLEANED_PAGES_PATH)

In [ ]:
preparation_summary = (
    cleaned_pages
    .groupby("document_id")
    .agg(
        pages=("page_number", "count"),
        raw_words=("raw_word_count", "sum"),
        clean_words=("clean_word_count", "sum"),
        empty_clean_pages=("clean_word_count", lambda x: int((x == 0).sum())),
        low_text_pages=("clean_word_count", lambda x: int((x < 20).sum()))
    )
    .reset_index()
)

preparation_summary = preparation_summary.merge(
    document_inventory[
        [
            "document_id",
            "document_name",
            "source_name",
            "document_type",
            "audience",
            "priority"
        ]
    ],
    on="document_id",
    how="left"
)

display(preparation_summary)

report = {
    "created_at": datetime.now().isoformat(),
    "raw_corpus_dir": RAW_CORPUS_DIR,
    "processed_corpus_dir": PROCESSED_CORPUS_DIR,
    "documents": preparation_summary.to_dict(orient="records"),
    "outputs": {
        "document_inventory": DOCUMENT_INVENTORY_PATH,
        "extracted_pages": EXTRACTED_PAGES_PATH,
        "cleaned_pages": CLEANED_PAGES_PATH
    }
}

with open(PREPARATION_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print("Saved:", PREPARATION_REPORT_PATH)

In [ ]:
required_paths = [
    DOCUMENT_INVENTORY_PATH,
    EXTRACTED_PAGES_PATH,
    CLEANED_PAGES_PATH,
    PREPARATION_REPORT_PATH
]

for path in required_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing output file: {path}")
    print("Found:", path)

if cleaned_pages["clean_word_count"].sum() == 0:
    raise ValueError("Cleaned corpus has zero words. Check PDF extraction.")

print("Step 4 complete.")

In [33]:
PROJECT_DIR = "/content/drive/MyDrive/llm"

PROCESSED_CORPUS_DIR = f"{PROJECT_DIR}/rag_corpus_processed"

DOCUMENT_INVENTORY_PATH = f"{PROCESSED_CORPUS_DIR}/document_inventory.csv"
CLEANED_PAGES_PATH = f"{PROCESSED_CORPUS_DIR}/cleaned_document_pages.csv"

SECTION_BLOCKS_PATH = f"{PROCESSED_CORPUS_DIR}/section_blocks.csv"
FINAL_RAG_CHUNKS_PATH = f"{PROCESSED_CORPUS_DIR}/final_rag_chunks.csv"
CHUNKING_REPORT_PATH = f"{PROCESSED_CORPUS_DIR}/final_chunking_report.csv"

print("Processed corpus directory:", PROCESSED_CORPUS_DIR)

Processed corpus directory: /content/drive/MyDrive/llm/rag_corpus_processed


In [ ]:
document_inventory = pd.read_csv(DOCUMENT_INVENTORY_PATH)
cleaned_pages = pd.read_csv(CLEANED_PAGES_PATH)

required_inventory_cols = [
    "document_id",
    "document_name",
    "source_name",
    "document_type",
    "audience",
    "year",
    "trust_level",
    "priority",
    "main_use"
]

required_page_cols = [
    "document_id",
    "file_name",
    "page_number",
    "clean_text",
    "clean_word_count"
]

missing_inventory = [c for c in required_inventory_cols if c not in document_inventory.columns]
missing_pages = [c for c in required_page_cols if c not in cleaned_pages.columns]

if missing_inventory:
    raise ValueError(f"Missing columns in document_inventory: {missing_inventory}")

if missing_pages:
    raise ValueError(f"Missing columns in cleaned_pages: {missing_pages}")

cleaned_pages["clean_text"] = cleaned_pages["clean_text"].fillna("")

print("Inventory:", document_inventory.shape)
print("Cleaned pages:", cleaned_pages.shape)

display(document_inventory)
display(cleaned_pages.head())

In [35]:
def normalize_line(line):
    line = str(line)
    line = re.sub(r"\s+", " ", line).strip()
    return line


def is_probably_noise_heading(line):
    low = line.lower().strip()

    noise_exact = {
        "contents",
        "table of contents",
        "my questions:",
        "how can i learn more?",
        "more resources",
        "cardiovascular conditions",
        "heart failure",
        "aha scientific statements"
    }

    if low in noise_exact:
        return True

    noise_contains = [
        "all rights reserved",
        "subject to notice of rights",
        "www.nice.org.uk",
        "heart.org/answersbyheart",
        "downloaded from",
        "doi:",
        "isbn:",
        "copyright"
    ]

    if any(x in low for x in noise_contains):
        return True

    return False


def heading_level_from_numbered_heading(line):
    """
    Examples:
    1. INTRODUCTION -> level 1
    4.1. Clinical Assessment -> level 2
    7.3.1. Renin-Angiotensin... -> level 3
    1.7 Starting and monitoring medication use -> level 2
    """

    line = normalize_line(line)

    m = re.match(r"^(\d+(?:\.\d+)*)(?:[a-z])?\.?\s+(.+)$", line)

    if not m:
        return None

    number_part = m.group(1)
    level = number_part.count(".") + 1

    return max(1, min(level, 5))


def is_title_like_heading(line):
    line = normalize_line(line)

    if len(line) < 4 or len(line) > 120:
        return False

    if line.endswith("."):
        return False

    if line.startswith(("•", "-", "－", "◦", "(", "[")):
        return False

    if re.fullmatch(r"[\d\s\.\-]+", line):
        return False

    words = line.split()

    if len(words) > 12:
        return False

    # Title-like short phrase.
    title_words = 0
    for w in words:
        clean_w = re.sub(r"[^A-Za-z]", "", w)
        if len(clean_w) > 2 and clean_w[:1].isupper():
            title_words += 1

    if title_words >= max(1, len(words) // 2):
        return True

    return False


def detect_heading(line, document_id):
    """
    Returns:
    {
        "is_heading": bool,
        "heading_text": str,
        "heading_level": int,
        "heading_type": str
    }
    """

    line = normalize_line(line)

    if not line:
        return {
            "is_heading": False,
            "heading_text": None,
            "heading_level": None,
            "heading_type": None
        }

    if is_probably_noise_heading(line):
        return {
            "is_heading": False,
            "heading_text": None,
            "heading_level": None,
            "heading_type": None
        }

    low = line.lower()

    # DOC001: AHA/ACC/HFSA guideline
    if document_id == "DOC001":
        if re.match(r"^(top 10 take-home messages|preamble|abstract)$", low):
            return {
                "is_heading": True,
                "heading_text": line,
                "heading_level": 1,
                "heading_type": "major_heading"
            }

        if re.match(r"^appendix\s+\d+", low):
            return {
                "is_heading": True,
                "heading_text": line,
                "heading_level": 1,
                "heading_type": "appendix_heading"
            }

        if re.match(r"^references$", low):
            return {
                "is_heading": True,
                "heading_text": line,
                "heading_level": 1,
                "heading_type": "references_heading"
            }

        numbered_level = heading_level_from_numbered_heading(line)
        if numbered_level is not None:
            return {
                "is_heading": True,
                "heading_text": line,
                "heading_level": numbered_level,
                "heading_type": "numbered_heading"
            }

        if line.startswith("Recommendations for "):
            return {
                "is_heading": True,
                "heading_text": line,
                "heading_level": 4,
                "heading_type": "recommendation_heading"
            }

        if line in ["Synopsis", "Recommendation-Specific Supportive Text", "Value Statement"]:
            return {
                "is_heading": True,
                "heading_text": line,
                "heading_level": 5,
                "heading_type": "supportive_heading"
            }

    # DOC002 / DOC003: NICE guidelines
    if document_id in ["DOC002", "DOC003"]:
        numbered_level = heading_level_from_numbered_heading(line)

        if numbered_level is not None:
            return {
                "is_heading": True,
                "heading_text": line,
                "heading_level": numbered_level,
                "heading_type": "numbered_heading"
            }

        nice_major = {
            "Overview",
            "Who is it for?",
            "Introduction",
            "Drug recommendations",
            "Key priorities for implementation",
            "Recommendations",
            "Recommendations for research",
            "Rationale and impact",
            "Context",
            "Finding more information and committee details",
            "Update information",
            "Terms used in this guideline"
        }

        if line in nice_major:
            return {
                "is_heading": True,
                "heading_text": line,
                "heading_level": 1,
                "heading_type": "major_heading"
            }

        nice_keywords = [
            "care",
            "diagnosis",
            "assessment",
            "monitoring",
            "treatment",
            "stabilisation",
            "stabilization",
            "pharmacological",
            "non-pharmacological",
            "medicine",
            "medication",
            "review",
            "kidney",
            "rehabilitation",
            "palliative",
            "discharge",
            "dopamine",
            "thiazide",
            "ultrafiltration",
            "ejection fraction",
            "heart failure"
        ]

        if is_title_like_heading(line) and any(k in low for k in nice_keywords):
            return {
                "is_heading": True,
                "heading_text": line,
                "heading_level": 3,
                "heading_type": "semantic_subheading"
            }

    # DOC004: AHA patient discharge packet
    if document_id == "DOC004":
        patient_major_headings = {
            "What is Heart Failure?",
            "Types of Heart Failure",
            "Your Ejection Fraction Explained",
            "How Can I Live With Heart Failure?",
            "Heart Failure Medications",
            "Lifestyle Changes",
            "Self-Check Plan",
            "Avoid Hidden Sources of Sodium",
            "Staying Active",
            "Keeping Your Appointments",
            "Charts & Logs"
        }

        if line in patient_major_headings:
            return {
                "is_heading": True,
                "heading_text": line,
                "heading_level": 1,
                "heading_type": "patient_packet_major_heading"
            }

        patient_keywords = [
            "medication",
            "medications",
            "sodium",
            "alcohol",
            "smoking",
            "weight",
            "exercise",
            "diet",
            "nutrition",
            "appointments",
            "warning",
            "symptoms",
            "fluid",
            "activity",
            "heart failure",
            "blood pressure",
            "stress"
        ]

        if is_title_like_heading(line) and any(k in low for k in patient_keywords):
            return {
                "is_heading": True,
                "heading_text": line,
                "heading_level": 2,
                "heading_type": "patient_packet_subheading"
            }

    return {
        "is_heading": False,
        "heading_text": None,
        "heading_level": None,
        "heading_type": None
    }

In [36]:
def update_heading_stack(stack, heading_text, heading_level):
    """
    Maintains a section hierarchy.

    Example:
    7. Stage C HF
    7.2. Diuretics and Decongestion Strategies
    Synopsis

    becomes:
    7. Stage C HF > 7.2. Diuretics and Decongestion Strategies > Synopsis
    """

    heading_level = int(heading_level)

    # Remove headings at the same or deeper level.
    stack = [item for item in stack if item["level"] < heading_level]

    stack.append({
        "level": heading_level,
        "title": heading_text
    })

    return stack


def stack_to_path(stack):
    if not stack:
        return "Document opening"

    return " > ".join([item["title"] for item in stack])

In [ ]:
metadata_by_doc = document_inventory.set_index("document_id").to_dict(orient="index")

section_blocks = []

for document_id, group in cleaned_pages.sort_values(["document_id", "page_number"]).groupby("document_id"):
    heading_stack = []

    current_block = None
    block_counter = 0

    for _, page_row in group.iterrows():
        page_number = int(page_row["page_number"])
        page_text = page_row["clean_text"]

        lines = [normalize_line(x) for x in str(page_text).splitlines()]
        lines = [x for x in lines if x]

        for line in lines:
            heading_info = detect_heading(line, document_id)

            if heading_info["is_heading"]:
                # Close previous block.
                if current_block is not None and len(current_block["items"]) > 0:
                    section_blocks.append(current_block)

                heading_stack = update_heading_stack(
                    heading_stack,
                    heading_info["heading_text"],
                    heading_info["heading_level"]
                )

                block_counter += 1

                current_block = {
                    "section_block_id": f"{document_id}_SEC{block_counter:04d}",
                    "document_id": document_id,
                    "section_title": heading_info["heading_text"],
                    "section_level": heading_info["heading_level"],
                    "heading_type": heading_info["heading_type"],
                    "section_path": stack_to_path(heading_stack),
                    "page_start": page_number,
                    "page_end": page_number,
                    "items": [
                        {
                            "page_number": page_number,
                            "text": heading_info["heading_text"]
                        }
                    ]
                }

            else:
                if current_block is None:
                    block_counter += 1

                    current_block = {
                        "section_block_id": f"{document_id}_SEC{block_counter:04d}",
                        "document_id": document_id,
                        "section_title": "Document opening",
                        "section_level": 0,
                        "heading_type": "document_opening",
                        "section_path": "Document opening",
                        "page_start": page_number,
                        "page_end": page_number,
                        "items": []
                    }

                current_block["items"].append({
                    "page_number": page_number,
                    "text": line
                })

                current_block["page_end"] = page_number

    if current_block is not None and len(current_block["items"]) > 0:
        section_blocks.append(current_block)


section_block_rows = []

for block in section_blocks:
    text = "\n".join([item["text"] for item in block["items"]])
    pages = [item["page_number"] for item in block["items"]]

    section_block_rows.append({
        "section_block_id": block["section_block_id"],
        "document_id": block["document_id"],
        "document_name": metadata_by_doc[block["document_id"]]["document_name"],
        "source_name": metadata_by_doc[block["document_id"]]["source_name"],
        "document_type": metadata_by_doc[block["document_id"]]["document_type"],
        "audience": metadata_by_doc[block["document_id"]]["audience"],
        "year": metadata_by_doc[block["document_id"]]["year"],
        "trust_level": metadata_by_doc[block["document_id"]]["trust_level"],
        "priority": metadata_by_doc[block["document_id"]]["priority"],
        "main_use": metadata_by_doc[block["document_id"]]["main_use"],
        "section_title": block["section_title"],
        "section_level": block["section_level"],
        "heading_type": block["heading_type"],
        "section_path": block["section_path"],
        "page_start": min(pages),
        "page_end": max(pages),
        "section_text": text,
        "section_word_count": len(text.split()),
        "section_char_count": len(text)
    })

section_blocks_df = pd.DataFrame(section_block_rows)

section_blocks_df.to_csv(SECTION_BLOCKS_PATH, index=False)

print("Section/subsection blocks:", section_blocks_df.shape)
display(section_blocks_df.head(20))

print("Saved:", SECTION_BLOCKS_PATH)

In [ ]:
for doc_id in sorted(section_blocks_df["document_id"].unique()):
    print("\n" + "=" * 120)
    print("Document:", doc_id)
    temp = section_blocks_df[section_blocks_df["document_id"] == doc_id]

    display(
        temp[
            [
                "section_block_id",
                "section_level",
                "heading_type",
                "section_title",
                "page_start",
                "page_end",
                "section_word_count"
            ]
        ].head(60)
    )

In [ ]:
def is_low_value_section(section_title, section_path, section_text, document_id):
    title = str(section_title).lower()
    path = str(section_path).lower()
    text = str(section_text).lower()
    word_count = len(str(section_text).split())

    if word_count < 40:
        return True

    low_value_terms = [
        "references",
        "appendix",
        "article information",
        "author relationships",
        "reviewer relationships",
        "presidents and staff",
        "finding more information and committee details",
        "update information",
        "terms used in this guideline",
        "copyright",
        "permissions"
    ]

    if any(term in title for term in low_value_terms):
        return True

    if any(term in path for term in low_value_terms):
        return True

    # Remove table-of-contents style sections.
    if title in ["contents", "table of contents"]:
        return True

    if text.count("................................................................") >= 3:
        return True

    return False


section_blocks_df["include_section"] = section_blocks_df.apply(
    lambda row: 0 if is_low_value_section(
        row["section_title"],
        row["section_path"],
        row["section_text"],
        row["document_id"]
    ) else 1,
    axis=1
)

print("Included sections:", int(section_blocks_df["include_section"].sum()))
print("Excluded sections:", int((section_blocks_df["include_section"] == 0).sum()))

display(
    section_blocks_df[
        [
            "document_id",
            "section_title",
            "page_start",
            "page_end",
            "section_word_count",
            "include_section"
        ]
    ].head(80)
)

In [40]:
def get_tail_items_by_word_count(items, overlap_words):
    tail_items = []
    total_words = 0

    for item in reversed(items):
        item_words = len(item["text"].split())

        if total_words + item_words > overlap_words and len(tail_items) > 0:
            break

        tail_items.insert(0, item)
        total_words += item_words

        if total_words >= overlap_words:
            break

    return tail_items


def split_section_items_into_chunks(items, target_words=520, min_words=80, overlap_words=80):
    """
    Splits a section/subsection into overlapping chunks while keeping page numbers.
    The final chunk boundaries are section-aware, not page-based.
    """

    chunks = []
    current_items = []
    current_words = 0

    for item in items:
        item_text = item["text"].strip()

        if not item_text:
            continue

        item_words = len(item_text.split())

        if current_items and current_words + item_words > target_words and current_words >= min_words:
            chunks.append(current_items)

            overlap_items = get_tail_items_by_word_count(current_items, overlap_words)

            current_items = overlap_items.copy()
            current_words = sum(len(x["text"].split()) for x in current_items)

        current_items.append(item)
        current_words += item_words

    if current_items:
        chunks.append(current_items)

    return chunks


def make_chunk_text(chunk_items):
    text = "\n".join([item["text"] for item in chunk_items])
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def build_retrieval_text(row, chunk_text):
    retrieval_text = f"""
Document name: {row["document_name"]}
Source: {row["source_name"]}
Document type: {row["document_type"]}
Audience: {row["audience"]}
Section path: {row["section_path"]}
Pages: {row["page_start"]}-{row["page_end"]}

Evidence text:
{chunk_text}
""".strip()

    return retrieval_text

In [ ]:
final_chunk_rows = []

for _, row in tqdm(section_blocks_df.iterrows(), total=len(section_blocks_df)):
    if int(row["include_section"]) != 1:
        continue

    section_text = str(row["section_text"])

    # Reconstruct section items with approximate page assignment.
    # Since section_blocks_df stores text only, we use the whole section page range here.
    # The source pages remain auditable at block level.
    section_items = []

    lines = [x.strip() for x in section_text.splitlines() if x.strip()]

    for line in lines:
        section_items.append({
            "page_number": int(row["page_start"]),
            "text": line
        })

    split_chunks = split_section_items_into_chunks(
        section_items,
        target_words=520,
        min_words=80,
        overlap_words=80
    )

    for chunk_idx, chunk_items in enumerate(split_chunks, start=1):
        chunk_text = make_chunk_text(chunk_items)

        if len(chunk_text.split()) < 40:
            continue

        chunk_id = f"{row['section_block_id']}_CH{chunk_idx:02d}"

        chunk_row = {
            "chunk_id": chunk_id,
            "section_block_id": row["section_block_id"],
            "document_id": row["document_id"],
            "document_name": row["document_name"],
            "source_name": row["source_name"],
            "document_type": row["document_type"],
            "audience": row["audience"],
            "year": row["year"],
            "trust_level": row["trust_level"],
            "priority": row["priority"],
            "main_use": row["main_use"],
            "section_title": row["section_title"],
            "section_level": row["section_level"],
            "heading_type": row["heading_type"],
            "section_path": row["section_path"],
            "page_start": row["page_start"],
            "page_end": row["page_end"],
            "chunk_index_in_section": chunk_idx,
            "chunk_text": chunk_text,
            "chunk_word_count": len(chunk_text.split()),
            "chunk_char_count": len(chunk_text),
        }

        chunk_row["retrieval_text"] = build_retrieval_text(chunk_row, chunk_text)

        final_chunk_rows.append(chunk_row)

final_rag_chunks = pd.DataFrame(final_chunk_rows)

if final_rag_chunks.empty:
    raise ValueError("No final RAG chunks were created. Check section detection and filtering.")

final_rag_chunks["include_in_index"] = 1

final_rag_chunks.to_csv(FINAL_RAG_CHUNKS_PATH, index=False)

print("Final RAG chunks:", final_rag_chunks.shape)
display(final_rag_chunks.head())

print("Saved:", FINAL_RAG_CHUNKS_PATH)

In [ ]:
chunking_report = (
    final_rag_chunks
    .groupby(["document_id", "document_name", "document_type", "audience"])
    .agg(
        final_chunks=("chunk_id", "count"),
        avg_words=("chunk_word_count", "mean"),
        min_words=("chunk_word_count", "min"),
        max_words=("chunk_word_count", "max"),
        pages_min=("page_start", "min"),
        pages_max=("page_end", "max")
    )
    .reset_index()
)

chunking_report["avg_words"] = chunking_report["avg_words"].round(2)

chunking_report.to_csv(CHUNKING_REPORT_PATH, index=False)

display(chunking_report)

print("Saved:", CHUNKING_REPORT_PATH)

In [ ]:
for doc_id in sorted(final_rag_chunks["document_id"].unique()):
    print("\n" + "=" * 120)
    print("Document:", doc_id)

    sample = final_rag_chunks[final_rag_chunks["document_id"] == doc_id].head(3)

    for _, row in sample.iterrows():
        print("\nChunk:", row["chunk_id"])
        print("Section path:", row["section_path"])
        print("Pages:", row["page_start"], "-", row["page_end"])
        print("Words:", row["chunk_word_count"])
        print(row["chunk_text"][:1500])

In [ ]:
required_chunk_cols = [
    "chunk_id",
    "document_id",
    "document_name",
    "source_name",
    "document_type",
    "audience",
    "section_title",
    "section_path",
    "page_start",
    "page_end",
    "chunk_text",
    "retrieval_text",
    "chunk_word_count",
    "include_in_index"
]

missing_cols = [c for c in required_chunk_cols if c not in final_rag_chunks.columns]

if missing_cols:
    raise ValueError(f"Missing required final chunk columns: {missing_cols}")

if final_rag_chunks["chunk_id"].duplicated().any():
    raise ValueError("Duplicate chunk IDs found.")

if final_rag_chunks["chunk_word_count"].min() < 40:
    raise ValueError("Some final chunks are too short.")

if final_rag_chunks["retrieval_text"].isna().any():
    raise ValueError("Some retrieval_text values are missing.")

doc_chunk_counts = final_rag_chunks["document_id"].value_counts().to_dict()

print("Final chunk counts by document:")
print(doc_chunk_counts)

for doc_id in ["DOC001", "DOC002", "DOC003", "DOC004"]:
    if doc_id not in doc_chunk_counts:
        print(f"Warning: {doc_id} has no final chunks.")

print("\nStep 5 complete.")
print("Your final RAG chunks are ready for embedding.")